# Specialization 1: Deep Learning — Ejercicios Prácticos

**Objetivos:**
- Entrenar red neuronal simple
- Visualizar backpropagation
- Implementar regularización

**Tiempo:** 45 min
**Dataset:** MNIST (dígitos manuscritos)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

## Ejercicio 1: Cargar y Preparar Datos

In [ ]:
# Cargar digits (8×8 imágenes de dígitos 0-9)
digits = load_digits()
X = digits.data
y = digits.target

print(f"Dataset shape: {X.shape}")
print(f"Dígitos: 0-9 ({len(np.unique(y))} clases)")
print(f"Imágenes: {len(X)}")

# Visualizar algunos dígitos
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].reshape(8, 8), cmap='gray')
    ax.set_title(f"Dígito: {y[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

## Ejercicio 2: Red Neuronal Simple (Shallow)

In [ ]:
# Arquitectura simple: 1 capa oculta
model_shallow = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(64,)),
    layers.Dense(10, activation='softmax')
])

model_shallow.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Arquitectura Shallow:")
model_shallow.summary()

# Entrenar
history_shallow = model_shallow.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

# Evaluar
loss, accuracy = model_shallow.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nTest Accuracy (Shallow): {accuracy:.4f}")

## Ejercicio 3: Red Neuronal Profunda (Deep)

In [ ]:
# Arquitectura profunda: múltiples capas
model_deep = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(64,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_deep.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Arquitectura Deep:")
model_deep.summary()

# Entrenar
history_deep = model_deep.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

# Evaluar
loss, accuracy = model_deep.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nTest Accuracy (Deep): {accuracy:.4f}")

## Ejercicio 4: Visualizar Learning Curves (Overfitting)

In [ ]:
# Comparar shallow vs deep
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Shallow
axes[0].plot(history_shallow.history['loss'], label='Train Loss')
axes[0].plot(history_shallow.history['val_loss'], label='Val Loss')
axes[0].set_title('Shallow Network')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Deep
axes[1].plot(history_deep.history['loss'], label='Train Loss')
axes[1].plot(history_deep.history['val_loss'], label='Val Loss')
axes[1].set_title('Deep Network')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Observar: ¿Dónde divergen train y validation en deep?")

## Ejercicio 5: Regularización (L2 + Dropout + Early Stopping)

In [ ]:
# Red con regularización
model_regularized = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(64,), 
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model_regularized.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Entrenar
history_reg = model_regularized.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

# Evaluar
loss, accuracy = model_regularized.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Accuracy (Regularized): {accuracy:.4f}")
print(f"Paró en época: {len(history_reg.history['loss'])}")

## Ejercicio 6: Comparación de Modelos

In [ ]:
# Evaluar los tres modelos
models = [
    ('Shallow', model_shallow),
    ('Deep', model_deep),
    ('Regularized', model_regularized)
]

results = []
for name, model in models:
    loss, acc = model.evaluate(X_test_scaled, y_test, verbose=0)
    results.append({'Model': name, 'Accuracy': acc})
    print(f"{name:15} → Accuracy: {acc:.4f}")

# Visualizar
import pandas as pd
df_results = pd.DataFrame(results)

plt.figure(figsize=(8, 5))
plt.bar(df_results['Model'], df_results['Accuracy'], color=['blue', 'orange', 'green'], edgecolor='black')
plt.ylabel('Test Accuracy')
plt.title('Comparación de Modelos')
plt.ylim([0.9, 1.0])
for i, v in enumerate(df_results['Accuracy']):
    plt.text(i, v + 0.005, f'{v:.4f}', ha='center')
plt.tight_layout()
plt.show()

## Resumen

In [ ]:
print("="*60)
print("RESUMEN: DEEP LEARNING")
print("="*60)
print()
print("1. ARQUITECTURAS:")
print("   - Shallow: rápido, puede underfittear")
print("   - Deep: más poder, pero riesgo de overfitting")
print()
print("2. REGULARIZACIÓN:")
print("   - L1/L2: penaliza pesos grandes")
print("   - Dropout: apaga neuronas aleatoriamente")
print("   - Early stopping: detén cuando val loss sube")
print()
print("3. VALIDACIÓN:")
print("   - Monitora train vs validation")
print("   - Gap grande = overfitting")
print()
print("="*60)